In [0]:
%sql
MERGE INTO my_catalog_pragati.gold_schema.RefinedMonthlySales AS tgt
USING (
    SELECT *
    FROM my_catalog_pragati.silver_schema.cleanedmonthlysales
    WHERE ingest_ts > (
        SELECT COALESCE(
            MAX(last_updt_ts),
            TO_TIMESTAMP('1990-01-01','yyyy-MM-dd')
        )
        FROM my_catalog_pragati.gold_schema.RefinedMonthlySales
    )
) AS src

ON tgt.sale_id = src.sale_id

WHEN MATCHED AND (
    tgt.product <> src.product OR
    tgt.category <> src.category OR
    tgt.quantity <> src.quantity OR
    tgt.price <> src.price OR
    tgt.sale_date <> src.sale_date OR
    tgt.region <> src.region
)

THEN UPDATE SET
    tgt.product = src.product,
    tgt.category = src.category,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.sale_date = src.sale_date,
    tgt.region = src.region,
    tgt.last_updt_ts = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
INSERT (
    sale_id,
    product,
    category,
    quantity,
    price,
    sale_date,
    region,
    initial_load_ts,
    last_updt_ts
)
VALUES (
    src.sale_id,
    src.product,
    src.category,
    src.quantity,
    src.price,
    src.sale_date,
    src.region,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
);